
# Customer Segmentation & Churn Analysis – Alfido Tech

**Goal:** Analyze customer transactions and behavior to identify customer segments, purchase patterns, and churn risks.

**Dataset:** `ecommerce_customer_data_custom_ratios.csv`

**Tasks Completed**
- Data Cleaning
- Feature Engineering
- Customer Segmentation (RFM + KMeans)
- Purchase Pattern Visualization
- Retention & Churn Analysis
- Business Recommendations


## 1. Import Libraries

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

sns.set()


## 2. Load Dataset

In [ ]:

df = pd.read_csv("ecommerce_customer_data_custom_ratios.csv")

print("Shape:", df.shape)
df.head()


## 3. Data Cleaning

In [ ]:

# Convert date column
df["Purchase Date"] = pd.to_datetime(df["Purchase Date"])

# Remove duplicates
df = df.drop_duplicates()

# Handle missing values
df = df.dropna()

print("After cleaning:", df.shape)


## 4. Basic Exploration

In [ ]:

df.describe()


In [ ]:

df.info()


## 5. Feature Engineering (RFM Metrics)

In [ ]:

snapshot_date = df["Purchase Date"].max() + pd.Timedelta(days=1)

rfm = df.groupby("Customer ID").agg({
    "Purchase Date": lambda x: (snapshot_date - x.max()).days,
    "Customer ID": "count",
    "Total Purchase Amount": "sum"
})

rfm.columns = ["Recency","Frequency","Monetary"]
rfm["Avg_Order_Value"] = rfm["Monetary"] / rfm["Frequency"]

rfm.head()


## 6. RFM Segmentation

In [ ]:

rfm["R_score"] = pd.qcut(rfm["Recency"],4,labels=[4,3,2,1])
rfm["F_score"] = pd.qcut(rfm["Frequency"].rank(method="first"),4,labels=[1,2,3,4])
rfm["M_score"] = pd.qcut(rfm["Monetary"],4,labels=[1,2,3,4])

rfm["RFM_Score"] = rfm[["R_score","F_score","M_score"]].astype(int).sum(axis=1)

rfm.head()


## 7. Customer Clustering (K-Means)

In [ ]:

rfm_cluster = rfm[["Recency","Frequency","Monetary"]]

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_cluster)

kmeans = KMeans(n_clusters=4, random_state=42)
rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)

rfm.head()


## 8. Cluster Visualization

In [ ]:

plt.figure()
sns.scatterplot(x=rfm["Frequency"], y=rfm["Monetary"], hue=rfm["Cluster"])
plt.title("Customer Segments by Frequency & Spending")
plt.show()


## 9. Purchase Pattern Analysis

In [ ]:

plt.figure()
sns.countplot(data=df, x="Product Category")
plt.xticks(rotation=45)
plt.title("Purchases by Category")
plt.show()


In [ ]:

plt.figure()
sns.countplot(data=df, x="Payment Method")
plt.title("Payment Method Distribution")
plt.show()


## 10. Age vs Spending

In [ ]:

plt.figure()
sns.scatterplot(data=df, x="Age", y="Total Purchase Amount")
plt.title("Age vs Spending")
plt.show()


## 11. Churn Analysis

In [ ]:

plt.figure()
sns.countplot(data=df, x="Churn")
plt.title("Churn Distribution")
plt.show()


In [ ]:

plt.figure()
sns.countplot(data=df, x="Gender", hue="Churn")
plt.title("Churn by Gender")
plt.show()


## 12. Key Insights


- High frequency customers generate the largest revenue.
- Customers with long gaps between purchases show higher churn risk.
- Certain product categories drive most transactions.
- Customers with higher spending should be prioritized for retention.
- Repeat buyers are the most valuable segment.


## 13. Business Recommendations for Alfido Tech


**1. Loyalty Program**
Reward frequent buyers with points and exclusive offers.

**2. Re-engagement Campaigns**
Send discounts or email reminders to inactive customers.

**3. Personalized Recommendations**
Recommend products based on previous purchase behavior.

**4. Improve Product Pages**
Reduce returns by adding better descriptions and reviews.

**5. VIP Program for High-Spenders**
Offer early product access and premium support.
